In [1]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

Layer Normalization

In [2]:
import torch
import torch.nn as nn

In [3]:
class LayerNorm(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(out_dim))
        self.shift = nn.Parameter(torch.zeros(out_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim = True)
        var = x.var(dim=-1, keepdim = True, unbiased = False)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)

        return self.scale * x_norm + self.shift

In [4]:
batch = torch.randn(2,5)
batch

layer = nn.Sequential(nn.Linear(5,6), nn.ReLU())
x = layer(batch)

In [5]:
torch.set_printoptions(sci_mode=False)

In [6]:
ln = LayerNorm(6)
out = ln(x)
out.mean(dim = -1)

tensor([-0.0000,  0.0000], grad_fn=<MeanBackward1>)

GeLU Activation Layer

In [7]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self,x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0/torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
            ))

In [8]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

In [25]:
inp = torch.rand(2, 3, 768)
ffn = FeedForward(GPT_CONFIG_124M)
out = ffn(inp)

Shortcut Connections

In [ ]:
class SampleDeepNeuralNetwork(nn.Module):

    def __init__(self, layer_sizes):
        super().__init__()

        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]), GELU()),
        ])

    def forward(self, x):
        for layer in self.layers:
            out = layer(x)
            #x = out without shortcut
            x = x + out #shortcut
        return x  
    
def print_gradients(model, x):
    output = model(x)
    target = torch.tensor([0.])
    loss = nn.MSELoss()
    loss = loss(output, target)
    loss.backward()
    for name, param in model.named_parameters():
        if "weight" in name:
            print(name, "has gradient mean: ", param.grad.abs().mean().item())


In [37]:
layer_sizes = [3,3,3,3,3,1]
dnn = SampleDeepNeuralNetwork(layer_sizes)

torch.manual_seed(123)
inp = torch.tensor([[1., 0., -1.]])
print_gradients(dnn, inp)


layers.0.0.weight has gradient mean:  0.7855343222618103
layers.1.0.weight has gradient mean:  0.7300012707710266
layers.2.0.weight has gradient mean:  1.0507218837738037
layers.3.0.weight has gradient mean:  0.8347376585006714
layers.4.0.weight has gradient mean:  2.2689902782440186


c:\Users\Jennifer\anaconda3\envs\LLMs\lib\site-packages\torch\nn\modules\loss.py:626: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 3])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Transformer Block

In [40]:
from package import MultiHeadAttention

class TransformerBlock(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.layer_norm1 = LayerNorm(cfg["emb_dim"])
        self.layer_norm2 = LayerNorm(cfg["emb_dim"])
        self.mha = MultiHeadAttention(cfg["emb_dim"], cfg["emb_dim"], cfg["context_length"], cfg["drop_rate"], cfg["n_heads"], cfg["qkv_bias"])
        self.dropout = nn.Dropout(cfg["drop_rate"])
        self.feed_forward = FeedForward(cfg)

    def forward(self, x):
        shortcut = x
        x = self.layer_norm1(x)
        x = self.mha(x)
        x = self.dropout(x)
        x = x + shortcut

        x = self.layer_norm2(x)
        x = self.feed_forward(x)
        x = self.dropout(x)
        x = x + shortcut

        return x


In [42]:
inp = torch.rand(2,3,768)
tb = TransformerBlock(GPT_CONFIG_124M)
out = tb(inp)
out.shape

torch.Size([2, 3, 768])